# Cardiac Ultrasound — Full Inference Pipeline

This notebook runs the **trained models** on new ultrasound images.

**You do NOT need the training dataset (database_nifti/) to use this notebook.**

### What you need (get all `.pt` files from your teammate):
| Weight File | Purpose |
|---|---|
| `segmentation_weights.pt` | U-Net segmentation (LV, myocardium, LA) |
| `cactus_resnet18_classification.pt` | Classify ultrasound view type (2CH, 4CH, etc.) |
| `cactus_resnet18_regression.pt` | Predict image quality score for each angle |
| `ef_trained.pt` | Predict Ejection Fraction directly |
| `multitask_ultrasound_model.pt` | Combined multi-task model |

- Your ultrasound images (`.nii.gz`, `.png`, `.jpg`, or `.dcm`)

### What this does:
1. **Classifies** what ultrasound view/angle the image shows
2. **Assesses quality** — is this a good angle?
3. **Segments** cardiac structures (LV cavity, myocardium, left atrium)
4. **Calculates metrics** (areas, volumes, EF, VTI, cardiac output)
5. **Visualizes** results with color overlays

In [6]:
# Cell [1] — Create a local virtual environment and install packages
import subprocess, sys, os

venv_path = os.path.expanduser("~/cardiac_venv")

# Create venv (only needs to run once)
if not os.path.exists(venv_path):
    subprocess.run([sys.executable, "-m", "venv", venv_path], check=True)
    print(f"✅ Created venv at {venv_path}")

# Install packages into the venv
pip_path = os.path.join(venv_path, "bin", "pip")
subprocess.run([pip_path, "install", "torch", "torchvision", "numpy", "nibabel", "matplotlib", "Pillow"], check=True)

# Add venv packages to this notebook's path
import site
site_packages = os.path.join(venv_path, "lib", f"python{sys.version_info.major}.{sys.version_info.minor}", "site-packages")
site.addsitedir(site_packages)

print(f"\n✅ All packages installed and available!")
print(f"   Location: {site_packages}")

✅ Created venv at /home/users/leah26/cardiac_venv
  Using cached nibabel-5.3.3-py3-none-any.whl.metadata (9.1 kB)
  Using cached filelock-3.24.2-py3-none-any.whl.metadata (2.0 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached fsspec-2026.2.0-py3-none-any.whl.metadata (10 kB)
  Using cached nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl.metadata (1.8 kB)
  Using cached nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl.metadata (1.7 kB)
  Using cached nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.m


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: /home/users/leah26/cardiac_venv/bin/python3.13 -m pip install --upgrade pip



✅ All packages installed and available!
   Location: /home/users/leah26/cardiac_venv/lib/python3.13/site-packages


In [7]:
# Cell [2] — Imports
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision.transforms import functional as F
from pathlib import Path

plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 11

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {DEVICE}")

Using device: cpu


In [ ]:
# Cell [3] — Configuration
# =====================================================
# UPDATE THESE PATHS to where YOU saved each .pt file
# =====================================================

WEIGHTS_DIR = "weights/"  # Folder containing all .pt files — CHANGE THIS

SEG_MODEL_PATH   = os.path.join(WEIGHTS_DIR, "segmentation_weights.pt")
CLASS_MODEL_PATH  = os.path.join(WEIGHTS_DIR, "cactus_resnet18_classification.pt")
REGR_MODEL_PATH   = os.path.join(WEIGHTS_DIR, "cactus_resnet18_regression.pt")
EF_MODEL_PATH     = os.path.join(WEIGHTS_DIR, "ef_trained.pt")
MULTI_MODEL_PATH  = os.path.join(WEIGHTS_DIR, "multitask_ultrasound_model.pt")

IMAGE_DIR = "test_images/"  # Folder with your ultrasound images — CHANGE THIS

IMG_SIZE = (256, 256)

LABEL_MAP = {
    0: 'Background',
    1: 'LV Cavity',
    2: 'Myocardium',
    3: 'Left Atrium'
}

print("Config loaded")
print(f"  Weights dir:  {WEIGHTS_DIR}")
print(f"  Image dir:    {IMAGE_DIR}")
print(f"  Input size:   {IMG_SIZE}")
print()
for name, path in [("Segmentation", SEG_MODEL_PATH),
                    ("Classification", CLASS_MODEL_PATH),
                    ("Regression", REGR_MODEL_PATH),
                    ("EF", EF_MODEL_PATH),
                    ("Multitask", MULTI_MODEL_PATH)]:
    found = os.path.exists(path)
    status = "FOUND" if found else "NOT FOUND"
    print(f"  {name:<16} {status:<12} {path}")

In [ ]:
# Cell [3b] — Inspect weight files to discover architectures
# Run this to see what's inside each .pt file so we know the correct model shapes

def inspect_weights(path, label=""):
    """Print summary of a .pt weight file's contents."""
    if not os.path.exists(path):
        print(f"  [{label}] File not found: {path}\n")
        return None

    data = torch.load(path, map_location="cpu", weights_only=False)
    print(f"  [{label}] {path}")
    print(f"    Top-level type: {type(data).__name__}")

    state_dict = None
    if isinstance(data, dict):
        print(f"    Keys: {list(data.keys())[:20]}")
        if "state_dict" in data:
            state_dict = data["state_dict"]
        elif "model_state_dict" in data:
            state_dict = data["model_state_dict"]
        elif all(isinstance(v, torch.Tensor) for v in list(data.values())[:5]):
            state_dict = data
    elif isinstance(data, nn.Module):
        print(f"    Saved as full model: {type(data).__name__}")
        state_dict = data.state_dict()

    if state_dict is not None:
        keys = list(state_dict.keys())
        print(f"    Total layers: {len(keys)}")
        print(f"    First 5 keys: {keys[:5]}")
        print(f"    Last 5 keys:  {keys[-5:]}")
        for k in keys[-5:]:
            print(f"      {k}: {state_dict[k].shape}")
    print()
    return data

print("=== Weight File Inspection ===\n")
for label, path in [("Segmentation", SEG_MODEL_PATH),
                     ("Classification", CLASS_MODEL_PATH),
                     ("Regression", REGR_MODEL_PATH),
                     ("EF", EF_MODEL_PATH),
                     ("Multitask", MULTI_MODEL_PATH)]:
    inspect_weights(path, label)

In [ ]:
# Cell [4] — Model Architectures
# UNet for segmentation + ResNet18 wrappers for classification/regression
# These must match exactly what was used during training.

import torchvision.models as models

# --- Segmentation: U-Net ---

class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=4):
        super(UNet, self).__init__()
        self.enc1 = self._conv_block(in_channels, 64)
        self.enc2 = self._conv_block(64, 128)
        self.enc3 = self._conv_block(128, 256)
        self.enc4 = self._conv_block(256, 512)
        self.bottleneck = self._conv_block(512, 1024)
        self.upconv4 = nn.ConvTranspose2d(1024, 512, 2, stride=2)
        self.dec4 = self._conv_block(1024, 512)
        self.upconv3 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec3 = self._conv_block(512, 256)
        self.upconv2 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec2 = self._conv_block(256, 128)
        self.upconv1 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec1 = self._conv_block(128, 64)
        self.out = nn.Conv2d(64, out_channels, 1)
        self.pool = nn.MaxPool2d(2)

    def _conv_block(self, in_ch, out_ch):
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))
        d4 = self.dec4(torch.cat([self.upconv4(b), e4], dim=1))
        d3 = self.dec3(torch.cat([self.upconv3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.upconv2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.upconv1(d2), e1], dim=1))
        return self.out(d1)


# --- Classification / Regression: ResNet18-based ---

def _make_resnet18(num_outputs, in_channels=1):
    """Build a ResNet18 with a custom first conv (grayscale) and final FC."""
    net = models.resnet18(weights=None)
    net.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
    net.fc = nn.Linear(net.fc.in_features, num_outputs)
    return net


def _detect_num_outputs(state_dict):
    """Auto-detect the number of outputs from the last FC layer in a state dict."""
    for key in ("fc.weight", "model.fc.weight", "classifier.weight"):
        if key in state_dict:
            return state_dict[key].shape[0]
    for key in reversed(list(state_dict.keys())):
        if "weight" in key and state_dict[key].ndim == 2:
            return state_dict[key].shape[0]
    return None


def load_resnet18_from_weights(path, fallback_outputs=1):
    """
    Load a ResNet18 model from a .pt file, auto-detecting the number of
    output classes/values from the saved state dict.
    """
    if not os.path.exists(path):
        print(f"  File not found: {path}")
        return None

    raw = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(raw, dict) and "state_dict" in raw:
        sd = raw["state_dict"]
    elif isinstance(raw, dict) and "model_state_dict" in raw:
        sd = raw["model_state_dict"]
    elif isinstance(raw, dict) and all(isinstance(v, torch.Tensor) for v in list(raw.values())[:3]):
        sd = raw
    else:
        print(f"  Unexpected format in {path} (type={type(raw).__name__}). "
              "Check Cell [3b] inspection output.")
        return None

    n_out = _detect_num_outputs(sd) or fallback_outputs
    net = _make_resnet18(n_out).to(DEVICE)
    net.load_state_dict(sd, strict=False)
    net.eval()
    return net, n_out


print(f"UNet defined ({sum(p.numel() for p in UNet().parameters()):,} params)")
print("ResNet18 builder defined (auto-detects output size from weights)")

In [ ]:
# Cell [5] — Load All Trained Models

loaded_models = {}

# 1. Segmentation (U-Net)
if os.path.exists(SEG_MODEL_PATH):
    seg_model = UNet().to(DEVICE)
    seg_sd = torch.load(SEG_MODEL_PATH, map_location=DEVICE, weights_only=False)
    if isinstance(seg_sd, dict) and "state_dict" in seg_sd:
        seg_sd = seg_sd["state_dict"]
    elif isinstance(seg_sd, dict) and "model_state_dict" in seg_sd:
        seg_sd = seg_sd["model_state_dict"]
    seg_model.load_state_dict(seg_sd, strict=False)
    seg_model.eval()
    loaded_models["segmentation"] = seg_model
    print(f"Segmentation model loaded from {SEG_MODEL_PATH}")
else:
    print(f"Segmentation weights not found at {SEG_MODEL_PATH}")

# 2. Classification (ResNet18 — view type)
if os.path.exists(CLASS_MODEL_PATH):
    result = load_resnet18_from_weights(CLASS_MODEL_PATH, fallback_outputs=15)
    if result:
        class_model, n_classes = result
        loaded_models["classification"] = class_model
        print(f"Classification model loaded ({n_classes} classes) from {CLASS_MODEL_PATH}")
else:
    print(f"Classification weights not found at {CLASS_MODEL_PATH}")

# 3. Regression (ResNet18 — quality score)
if os.path.exists(REGR_MODEL_PATH):
    result = load_resnet18_from_weights(REGR_MODEL_PATH, fallback_outputs=1)
    if result:
        regr_model, n_out = result
        loaded_models["regression"] = regr_model
        print(f"Regression model loaded ({n_out} outputs) from {REGR_MODEL_PATH}")
else:
    print(f"Regression weights not found at {REGR_MODEL_PATH}")

# 4. EF model
if os.path.exists(EF_MODEL_PATH):
    result = load_resnet18_from_weights(EF_MODEL_PATH, fallback_outputs=1)
    if result:
        ef_model, n_out = result
        loaded_models["ef"] = ef_model
        print(f"EF model loaded ({n_out} outputs) from {EF_MODEL_PATH}")
    else:
        print(f"EF weights format not recognized — check Cell [3b] inspection output")
else:
    print(f"EF weights not found at {EF_MODEL_PATH}")

# 5. Multitask model
if os.path.exists(MULTI_MODEL_PATH):
    print(f"Multitask weights found at {MULTI_MODEL_PATH}")
    print("  (Architecture unknown — check Cell [3b] inspection output to determine how to load it)")
else:
    print(f"Multitask weights not found at {MULTI_MODEL_PATH}")

print(f"\nLoaded {len(loaded_models)} model(s): {list(loaded_models.keys())}")

In [ ]:
# Cell [6] — Image Loading Helpers
# Supports multiple formats your ultrasound hardware might output

def load_ultrasound_image(path):
    """
    Load an ultrasound image from various formats.
    Returns: (image_array, pixel_spacing)
        image_array: 2D numpy array (H, W)
        pixel_spacing: (dx, dy) in mm, or (1.0, 1.0) if unknown
    """
    path = str(path)
    ext = path.lower()
    
    if ext.endswith(('.nii', '.nii.gz')):
        # NIfTI format (CAMUS dataset format)
        import nibabel as nib
        nii = nib.load(path)
        img = nii.get_fdata().astype(np.float32)
        # Handle 3D volumes — take middle slice if needed
        if img.ndim == 3:
            img = img[:, :, img.shape[2] // 2]
        spacing = nii.header.get_zooms()[:2]
        return img, spacing
    
    elif ext.endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tiff')):
        # Standard image formats
        from PIL import Image
        img = np.array(Image.open(path).convert('L')).astype(np.float32)
        return img, (1.0, 1.0)  # Unknown pixel spacing
    
    elif ext.endswith('.dcm'):
        # DICOM format
        try:
            import pydicom
            ds = pydicom.dcmread(path)
            img = ds.pixel_array.astype(np.float32)
            spacing = getattr(ds, 'PixelSpacing', [1.0, 1.0])
            return img, (float(spacing[0]), float(spacing[1]))
        except ImportError:
            raise ImportError("Install pydicom: pip install pydicom")
    
    elif ext.endswith('.npy'):
        # NumPy array
        img = np.load(path).astype(np.float32)
        return img, (1.0, 1.0)
    
    else:
        raise ValueError(f"Unsupported format: {path}")

print("✅ Image loader ready")
print("   Supported formats: .nii.gz, .nii, .png, .jpg, .dcm, .npy")

In [ ]:
# Cell [7] — Segmentation Inference

def predict_segmentation(seg_model, image_array):
    """
    Run U-Net segmentation on a single 2D ultrasound image.

    Args:
        seg_model: loaded UNet model
        image_array: 2D numpy array (H, W)

    Returns:
        pred_mask: 2D numpy array (H, W) with labels 0-3
        confidence: 2D numpy array (H, W) with prediction confidence
    """
    original_shape = image_array.shape[:2]

    img = image_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)

    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
    img_tensor = F.resize(img_tensor, IMG_SIZE, interpolation=F.InterpolationMode.BILINEAR)
    img_tensor = img_tensor.to(DEVICE)

    with torch.no_grad():
        output = seg_model(img_tensor)
        probs = torch.softmax(output, dim=1)
        confidence, pred = torch.max(probs, dim=1)

    pred = F.resize(pred.unsqueeze(1).float(), original_shape,
                    interpolation=F.InterpolationMode.NEAREST)
    confidence = F.resize(confidence.unsqueeze(1).float(), original_shape,
                          interpolation=F.InterpolationMode.BILINEAR)

    return (
        pred.squeeze().cpu().numpy().astype(np.int64),
        confidence.squeeze().cpu().numpy(),
    )

print("Segmentation inference function ready")

In [ ]:
# Cell [7b] — Classification & Regression Inference

def predict_view_class(image_array):
    """
    Classify ultrasound view type using the classification model.
    Returns (predicted_class_index, class_probabilities).
    """
    if "classification" not in loaded_models:
        return None, None

    img = image_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
    img_tensor = F.resize(img_tensor, IMG_SIZE, interpolation=F.InterpolationMode.BILINEAR)
    img_tensor = img_tensor.to(DEVICE)

    with torch.no_grad():
        logits = loaded_models["classification"](img_tensor)
        probs = torch.softmax(logits, dim=1).squeeze().cpu().numpy()
    return int(probs.argmax()), probs


def predict_quality_score(image_array):
    """
    Predict image quality score using the regression model.
    Returns a float score (interpretation depends on training).
    """
    if "regression" not in loaded_models:
        return None

    img = image_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
    img_tensor = F.resize(img_tensor, IMG_SIZE, interpolation=F.InterpolationMode.BILINEAR)
    img_tensor = img_tensor.to(DEVICE)

    with torch.no_grad():
        score = loaded_models["regression"](img_tensor)
    return float(score.squeeze().cpu())


def predict_ef_direct(image_array):
    """
    Predict Ejection Fraction directly using the EF model.
    Returns a float EF value.
    """
    if "ef" not in loaded_models:
        return None

    img = image_array.astype(np.float32)
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img_tensor = torch.from_numpy(img).unsqueeze(0).unsqueeze(0)
    img_tensor = F.resize(img_tensor, IMG_SIZE, interpolation=F.InterpolationMode.BILINEAR)
    img_tensor = img_tensor.to(DEVICE)

    with torch.no_grad():
        ef_val = loaded_models["ef"](img_tensor)
    return float(ef_val.squeeze().cpu())


available = []
if "classification" in loaded_models:
    available.append("View classification")
if "regression" in loaded_models:
    available.append("Quality regression")
if "ef" in loaded_models:
    available.append("Direct EF prediction")
print(f"Inference functions ready: {available if available else 'Only segmentation (other models not loaded)'}")

In [ ]:
# Cell [8] — Cardiac Metrics Calculation

def calculate_areas(pred_mask, pixel_spacing_mm=(1.0, 1.0)):
    """Calculate areas of each segmented structure in mm²."""
    dx, dy = pixel_spacing_mm
    pixel_area = dx * dy
    
    return {
        'lv_area_mm2': float((pred_mask == 1).sum() * pixel_area),
        'myocardium_area_mm2': float((pred_mask == 2).sum() * pixel_area),
        'la_area_mm2': float((pred_mask == 3).sum() * pixel_area),
    }


def calculate_lv_length_mm(pred_mask, pixel_spacing_mm=(1.0, 1.0)):
    """Estimate LV long-axis length from segmentation mask."""
    dx, dy = pixel_spacing_mm
    ys, xs = np.where(pred_mask == 1)
    
    if len(xs) == 0:
        return None
    
    length_pixels = np.sqrt((xs.max() - xs.min())**2 + (ys.max() - ys.min())**2)
    return float(length_pixels * np.mean([dx, dy]))


def calculate_volume_biplane(area_2ch_mm2, area_4ch_mm2, length_mm):
    """
    Biplane area-length method (modified Simpson's):
    V = (8 / 3π) × (A_2CH × A_4CH) / L
    Returns volume in mL.
    """
    if None in [area_2ch_mm2, area_4ch_mm2, length_mm] or length_mm == 0:
        return None
    
    volume_mm3 = (8 / (3 * np.pi)) * (area_2ch_mm2 * area_4ch_mm2) / length_mm
    return volume_mm3 / 1000  # mm³ → mL


def calculate_ef(edv_ml, esv_ml):
    """Ejection Fraction (%)."""
    if edv_ml is None or esv_ml is None or edv_ml == 0:
        return None
    return ((edv_ml - esv_ml) / edv_ml) * 100


def calculate_cardiac_output(sv_ml, heart_rate_bpm):
    """Cardiac Output in L/min."""
    return (sv_ml * heart_rate_bpm) / 1000


def calculate_vti(sv_ml, lvot_diameter_cm=2.0):
    """
    Velocity Time Integral (cm).
    VTI = SV / LVOT_area
    """
    lvot_area = np.pi * (lvot_diameter_cm / 2)**2
    return sv_ml / lvot_area  # mL = cm³, so result is in cm


print("✅ Cardiac metrics functions ready")

In [ ]:
# Cell [9] — Visualization

def visualize_result(image, pred_mask, confidence, title=""):
    """Display original image, segmentation, and overlay."""
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    
    # Original
    axes[0].imshow(image, cmap='gray')
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Segmentation mask
    axes[1].imshow(pred_mask, cmap='tab10', vmin=0, vmax=3)
    axes[1].set_title('Segmentation')
    axes[1].axis('off')
    
    # Overlay
    axes[2].imshow(image, cmap='gray')
    for label, color, name in [(1, 'Reds', 'LV'), (2, 'Blues', 'Myo'), (3, 'Greens', 'LA')]:
        mask_overlay = np.ma.masked_where(pred_mask != label, pred_mask)
        axes[2].imshow(mask_overlay, cmap=color, alpha=0.5, vmin=0, vmax=3)
    axes[2].set_title('Overlay (R=LV, B=Myo, G=LA)')
    axes[2].axis('off')
    
    # Confidence map
    im = axes[3].imshow(confidence, cmap='RdYlGn', vmin=0.5, vmax=1.0)
    axes[3].set_title(f'Confidence (avg: {confidence.mean():.2f})')
    axes[3].axis('off')
    plt.colorbar(im, ax=axes[3], fraction=0.046)
    
    if title:
        plt.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    plt.show()

print("✅ Visualization functions ready")

In [ ]:
# Cell [10] — Run on a Single Image (all models)
# UPDATE THIS PATH to your ultrasound image

test_image_path = "test_images/your_ultrasound.nii.gz"  # <-- CHANGE THIS

if os.path.exists(test_image_path):
    image, spacing = load_ultrasound_image(test_image_path)
    print(f"Loaded: {test_image_path}")
    print(f"  Shape: {image.shape}, Pixel spacing: {spacing} mm")

    # 1 — View classification
    view_idx, view_probs = predict_view_class(image)
    if view_idx is not None:
        print(f"\n[Classification] Predicted view index: {view_idx}  (confidence: {view_probs[view_idx]:.2f})")
    else:
        print("\n[Classification] Model not loaded — skipped")

    # 2 — Quality score
    quality = predict_quality_score(image)
    if quality is not None:
        print(f"[Quality]        Score: {quality:.3f}")
    else:
        print("[Quality]        Model not loaded — skipped")

    # 3 — Segmentation
    if "segmentation" in loaded_models:
        pred_mask, confidence = predict_segmentation(loaded_models["segmentation"], image)
        areas = calculate_areas(pred_mask, spacing)
        lv_length = calculate_lv_length_mm(pred_mask, spacing)
        print(f"\n[Segmentation]")
        print(f"  LV Cavity Area:    {areas['lv_area_mm2']:.1f} mm2")
        print(f"  Myocardium Area:   {areas['myocardium_area_mm2']:.1f} mm2")
        print(f"  Left Atrium Area:  {areas['la_area_mm2']:.1f} mm2")
        if lv_length:
            print(f"  LV Length:         {lv_length:.1f} mm")
        print(f"  Avg Confidence:    {confidence.mean():.2f}")
        visualize_result(image, pred_mask, confidence, title=Path(test_image_path).name)
    else:
        print("\n[Segmentation]   Model not loaded — skipped")

    # 4 — Direct EF prediction
    ef_direct = predict_ef_direct(image)
    if ef_direct is not None:
        print(f"[EF Direct]      Predicted EF: {ef_direct:.1f}%")
    else:
        print("[EF Direct]      Model not loaded — skipped")
else:
    print(f"File not found: {test_image_path}")
    print(f"Update 'test_image_path' above to point to your ultrasound image")

In [ ]:
# Cell [11] — Full Pipeline: Process a Set of 15 Probe Images (all models)

def process_all_images(image_dir):
    """
    Run every loaded model on all ultrasound images in a directory.
    Returns a list of per-image result dicts.
    """
    supported_ext = ('.nii.gz', '.nii', '.png', '.jpg', '.jpeg', '.dcm', '.npy')

    image_files = []
    for f in sorted(os.listdir(image_dir)):
        if any(f.lower().endswith(ext) for ext in supported_ext):
            image_files.append(os.path.join(image_dir, f))

    if not image_files:
        print(f"No supported images found in {image_dir}")
        return []

    print(f"Found {len(image_files)} images in {image_dir}\n")

    results = []
    for i, img_path in enumerate(image_files, 1):
        fname = Path(img_path).name
        print(f"[{i}/{len(image_files)}] {fname}")

        try:
            image, spacing = load_ultrasound_image(img_path)
            entry = {
                'file': fname,
                'image': image,
                'spacing': spacing,
            }

            # Classification
            view_idx, view_probs = predict_view_class(image)
            entry['view_class'] = view_idx
            entry['view_probs'] = view_probs

            # Quality
            entry['quality_score'] = predict_quality_score(image)

            # Segmentation
            if "segmentation" in loaded_models:
                pred_mask, confidence = predict_segmentation(loaded_models["segmentation"], image)
                areas = calculate_areas(pred_mask, spacing)
                entry.update({
                    'pred_mask': pred_mask,
                    'confidence': confidence,
                    'areas': areas,
                    'lv_length_mm': calculate_lv_length_mm(pred_mask, spacing),
                    'avg_confidence': float(confidence.mean()),
                    'has_lv': bool((pred_mask == 1).any()),
                    'has_myocardium': bool((pred_mask == 2).any()),
                    'has_la': bool((pred_mask == 3).any()),
                })
                seg_info = f"LV={areas['lv_area_mm2']:.0f}mm2"
            else:
                seg_info = "seg N/A"

            # Direct EF
            entry['ef_direct'] = predict_ef_direct(image)

            qual_str = f"Q={entry['quality_score']:.2f}" if entry['quality_score'] is not None else "Q=N/A"
            view_str = f"View={view_idx}" if view_idx is not None else "View=N/A"
            ef_str = f"EF={entry['ef_direct']:.1f}%" if entry['ef_direct'] is not None else "EF=N/A"
            print(f"   {view_str} | {qual_str} | {seg_info} | {ef_str}")

            results.append(entry)

        except Exception as e:
            print(f"   Error: {e}")
            results.append({'file': fname, 'error': str(e)})

    return results

if os.path.isdir(IMAGE_DIR):
    all_results = process_all_images(IMAGE_DIR)
else:
    print(f"Directory not found: {IMAGE_DIR}")
    print(f"Create the folder and add your ultrasound images, or update IMAGE_DIR in Cell [3]")
    all_results = []

In [ ]:
# Cell [12] — Full Cardiac Report (all models)

def compute_full_cardiac_report(results, heart_rate_bpm=75, lvot_diameter_cm=2.0):
    """
    Print a comprehensive report using all available model outputs.
    """
    valid = [r for r in results if 'error' not in r]

    if not valid:
        print("No valid results to analyze")
        return None

    print("\n" + "=" * 80)
    print("  CARDIAC REPORT")
    print("=" * 80)

    # --- Per-image table ---
    has_class = any(r.get('view_class') is not None for r in valid)
    has_qual  = any(r.get('quality_score') is not None for r in valid)
    has_seg   = any('areas' in r for r in valid)
    has_ef    = any(r.get('ef_direct') is not None for r in valid)

    header = f"{'#':<3} {'File':<30}"
    if has_class: header += f" {'View':>5}"
    if has_qual:  header += f" {'Qual':>6}"
    if has_seg:   header += f" {'LV mm2':>8} {'Myo mm2':>8} {'LA mm2':>8} {'Conf':>6}"
    if has_ef:    header += f" {'EF%':>6}"
    print(f"\n{header}")
    print("-" * len(header))

    for i, r in enumerate(valid):
        row = f"{i:<3} {r['file']:<30}"
        if has_class:
            v = r.get('view_class')
            row += f" {v if v is not None else '-':>5}"
        if has_qual:
            q = r.get('quality_score')
            row += f" {q:>6.2f}" if q is not None else f" {'-':>6}"
        if has_seg and 'areas' in r:
            row += (f" {r['areas']['lv_area_mm2']:>8.1f}"
                    f" {r['areas']['myocardium_area_mm2']:>8.1f}"
                    f" {r['areas']['la_area_mm2']:>8.1f}"
                    f" {r.get('avg_confidence', 0):>6.2f}")
        elif has_seg:
            row += f" {'—':>8} {'—':>8} {'—':>8} {'—':>6}"
        if has_ef:
            ef = r.get('ef_direct')
            row += f" {ef:>6.1f}" if ef is not None else f" {'—':>6}"
        print(row)

    # --- Volume / EF / CO / VTI section ---
    print("\n" + "-" * 80)
    print("To compute volumes, EF, VTI, and cardiac output from segmentation,")
    print("identify which images correspond to 2CH/4CH at ED/ES phases.")
    print("Then uncomment and update the indices below:\n")
    print("# idx_2ch_ed = 0   # index in 'valid' for 2-chamber end-diastole")
    print("# idx_2ch_es = 1   # 2-chamber end-systole")
    print("# idx_4ch_ed = 2   # 4-chamber end-diastole")
    print("# idx_4ch_es = 3   # 4-chamber end-systole")
    print("#")
    print("# a2ch_ed = valid[idx_2ch_ed]['areas']['lv_area_mm2']")
    print("# a4ch_ed = valid[idx_4ch_ed]['areas']['lv_area_mm2']")
    print("# l_ed    = valid[idx_4ch_ed]['lv_length_mm']")
    print("# edv     = calculate_volume_biplane(a2ch_ed, a4ch_ed, l_ed)")
    print("#")
    print("# a2ch_es = valid[idx_2ch_es]['areas']['lv_area_mm2']")
    print("# a4ch_es = valid[idx_4ch_es]['areas']['lv_area_mm2']")
    print("# l_es    = valid[idx_4ch_es]['lv_length_mm']")
    print("# esv     = calculate_volume_biplane(a2ch_es, a4ch_es, l_es)")
    print("#")
    print("# sv  = edv - esv")
    print("# ef  = calculate_ef(edv, esv)")
    print("# co  = calculate_cardiac_output(sv, heart_rate_bpm)")
    print("# vti = calculate_vti(sv, lvot_diameter_cm)")

    return valid

if all_results:
    report = compute_full_cardiac_report(all_results, heart_rate_bpm=75, lvot_diameter_cm=2.0)

In [ ]:
# Cell [13] — Visualize All Results in a Grid

def visualize_all_results(results, cols=5):
    """Display all segmentation results in a grid."""
    valid = [r for r in results if 'error' not in r]
    
    if not valid:
        print("No valid results to display")
        return
    
    n = len(valid)
    rows = (n + cols - 1) // cols
    
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1) if cols > 1 else np.array([[axes]])
    
    for i, r in enumerate(valid):
        row, col = i // cols, i % cols
        ax = axes[row, col]
        
        ax.imshow(r['image'], cmap='gray')
        for label, cmap_name in [(1, 'Reds'), (2, 'Blues'), (3, 'Greens')]:
            overlay = np.ma.masked_where(r['pred_mask'] != label, r['pred_mask'])
            ax.imshow(overlay, cmap=cmap_name, alpha=0.5, vmin=0, vmax=3)
        
        ax.set_title(f"{r['file']}\nConf: {r['avg_confidence']:.2f}", fontsize=9)
        ax.axis('off')
    
    # Hide empty subplots
    for i in range(n, rows * cols):
        row, col = i // cols, i % cols
        axes[row, col].axis('off')
    
    plt.suptitle("Segmentation Results — All Images", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

if all_results:
    visualize_all_results(all_results)